# Gold Layer — Star Schema

Esta capa construye el modelo dimensional final del proyecto Wanderbricks,
listo para ser consumido desde Power BI.

Siguiendo el diseño aprobado en el Informe de Avance, el modelo está
compuesto por:

**Tabla de hechos**

- `gold_fact_reservas`

**Tablas de dimensiones**

- `gold_dim_users`
- `gold_dim_properties`
- `gold_dim_destinations`
- `gold_dim_time`

Todas las tablas se construyen a partir de Silver, manteniendo el principio
de que Gold solo lee de Silver y nunca de Bronze.

## gold_dim_users

Dimensión de usuarios. Se conservan los atributos descriptivos relevantes
para análisis (país, tipo de usuario, perfil empresarial).

In [ ]:
%sql
CREATE OR REPLACE TABLE gold.gold_dim_users AS
SELECT
  user_id              AS user_key,
  user_id,
  name,
  email,
  country,
  user_type,
  is_business,
  company_name,
  created_at           AS registration_date
FROM silver.silver_users;

In [ ]:
%sql
SELECT * FROM gold.gold_dim_users LIMIT 5;

## gold_dim_properties

Dimensión de propiedades. Se omite la columna `description` (texto largo)
para mantener la dimensión liviana para los dashboards.

In [ ]:
%sql
CREATE OR REPLACE TABLE gold.gold_dim_properties AS
SELECT
  property_id              AS property_key,
  property_id,
  host_id,
  destination_id,
  title,
  property_type,
  max_guests,
  bedrooms,
  bathrooms,
  base_price,
  property_latitude,
  property_longitude
FROM silver.silver_properties;

In [ ]:
%sql
SELECT * FROM gold.gold_dim_properties LIMIT 5;

## gold_dim_destinations

Dimensión geográfica de destinos. Se descarta el campo `description`
(markdown extenso, no aporta al análisis dimensional).

In [ ]:
%sql
CREATE OR REPLACE TABLE gold.gold_dim_destinations AS
SELECT
  destination_id              AS destination_key,
  destination_id,
  destination                 AS destination_name,
  country,
  state_or_province,
  state_or_province_code
FROM silver.silver_destinations;

In [ ]:
%sql
SELECT * FROM gold.gold_dim_destinations LIMIT 5;

## gold_dim_time

Dimensión temporal generada dinámicamente a partir del rango de fechas
observado en las reservas (`MIN(check_in)` y `MAX(check_out)`).

Incluye atributos típicos para análisis temporal en Power BI:
año, trimestre, mes, día de la semana, semana del año y bandera de fin de semana.

In [ ]:
%sql
CREATE OR REPLACE TABLE gold.gold_dim_time AS
WITH bounds AS (
  SELECT
    MIN(check_in)  AS min_date,
    MAX(check_out) AS max_date
  FROM silver.silver_bookings
),
dates AS (
  SELECT
    EXPLODE(
      SEQUENCE(min_date, max_date, INTERVAL 1 DAY)
    ) AS date
  FROM bounds
)
SELECT
  date                                  AS tiempo_id,
  date,
  YEAR(date)                            AS year,
  QUARTER(date)                         AS quarter,
  MONTH(date)                           AS month,
  DATE_FORMAT(date, 'MMMM')             AS month_name,
  DAY(date)                             AS day,
  DAYOFWEEK(date)                       AS day_of_week,
  DATE_FORMAT(date, 'EEEE')             AS day_name,
  WEEKOFYEAR(date)                      AS week_of_year,
  CASE
    WHEN DAYOFWEEK(date) IN (1, 7) THEN true
    ELSE false
  END                                   AS is_weekend
FROM dates;

In [ ]:
%sql
SELECT * FROM gold.gold_dim_time ORDER BY date LIMIT 5;

## gold_fact_reservas

Tabla de hechos central del Star Schema.

**Granularidad**: una fila por reserva (`booking_id`).

**Métricas**:

- `total_amount` (importe total de la reserva)
- `total_nights` (noches reservadas)
- `cantidad_reservas` (constante 1, para sumarizar reservas)
- `payment_amount` (suma de pagos completados asociados a la reserva)

**Llaves foráneas**: `user_id`, `property_id`, `destination_id`, `tiempo_id`.

El `destination_id` se obtiene mediante JOIN con `silver_properties`,
ya que `bookings` no lo contiene directamente.

Los pagos se pre-agregan por `booking_id` (status `completed`) para
evitar duplicación cuando una reserva tiene múltiples transacciones.

In [ ]:
%sql
CREATE OR REPLACE TABLE gold.gold_fact_reservas AS
WITH pagos_agregados AS (
  SELECT
    booking_id,
    SUM(amount) AS payment_amount
  FROM silver.silver_payments
  WHERE status = 'completed'
  GROUP BY booking_id
)
SELECT
  b.booking_id,
  b.user_id,
  b.property_id,
  p.destination_id,
  b.check_in                       AS tiempo_id,
  b.total_amount,
  b.total_nights,
  1                                AS cantidad_reservas,
  COALESCE(pa.payment_amount, 0)   AS payment_amount,
  b.status                         AS booking_status,
  b.check_in,
  b.check_out,
  b.guests_count
FROM silver.silver_bookings b
LEFT JOIN silver.silver_properties p
  ON b.property_id = p.property_id
LEFT JOIN pagos_agregados pa
  ON b.booking_id = pa.booking_id;

In [ ]:
%sql
SELECT * FROM gold.gold_fact_reservas LIMIT 5;

## Validaciones del Star Schema

Se verifica que las llaves foráneas de la fact table apunten correctamente
a las dimensiones (no haya huérfanos).

In [ ]:
%sql
-- Conteo general de filas en cada tabla del modelo
SELECT 'gold_fact_reservas'   AS tabla, COUNT(*) AS registros FROM gold.gold_fact_reservas
UNION ALL
SELECT 'gold_dim_users'       AS tabla, COUNT(*) AS registros FROM gold.gold_dim_users
UNION ALL
SELECT 'gold_dim_properties'  AS tabla, COUNT(*) AS registros FROM gold.gold_dim_properties
UNION ALL
SELECT 'gold_dim_destinations' AS tabla, COUNT(*) AS registros FROM gold.gold_dim_destinations
UNION ALL
SELECT 'gold_dim_time'        AS tabla, COUNT(*) AS registros FROM gold.gold_dim_time
ORDER BY tabla;

In [ ]:
%sql
-- Reservas sin usuario en la dimensión (debe dar 0)
SELECT COUNT(*) AS reservas_sin_usuario
FROM gold.gold_fact_reservas f
LEFT JOIN gold.gold_dim_users d
  ON f.user_id = d.user_key
WHERE d.user_key IS NULL;

In [ ]:
%sql
-- Reservas sin propiedad en la dimensión (debe dar 0)
SELECT COUNT(*) AS reservas_sin_propiedad
FROM gold.gold_fact_reservas f
LEFT JOIN gold.gold_dim_properties d
  ON f.property_id = d.property_key
WHERE d.property_key IS NULL;

In [ ]:
%sql
-- Reservas sin destino en la dimensión (puede dar > 0 si hay propiedades sin destination_id)
SELECT COUNT(*) AS reservas_sin_destino
FROM gold.gold_fact_reservas f
LEFT JOIN gold.gold_dim_destinations d
  ON f.destination_id = d.destination_key
WHERE d.destination_key IS NULL;

## Métricas de negocio listas para Power BI

Consultas de ejemplo que demuestran que el Star Schema responde
las preguntas analíticas planteadas en el Informe de Avance.

In [ ]:
%sql
-- Ingresos totales por país de destino
SELECT
  d.country,
  ROUND(SUM(f.total_amount), 2) AS ingresos_totales,
  COUNT(*)                       AS total_reservas
FROM gold.gold_fact_reservas f
JOIN gold.gold_dim_destinations d
  ON f.destination_id = d.destination_key
GROUP BY d.country
ORDER BY ingresos_totales DESC
LIMIT 10;

In [ ]:
%sql
-- Reservas por mes y año
SELECT
  t.year,
  t.month,
  t.month_name,
  COUNT(*)                       AS total_reservas,
  ROUND(SUM(f.total_amount), 2)  AS ingresos
FROM gold.gold_fact_reservas f
JOIN gold.gold_dim_time t
  ON f.tiempo_id = t.tiempo_id
GROUP BY t.year, t.month, t.month_name
ORDER BY t.year, t.month;

In [ ]:
%sql
-- Top 10 propiedades por ingresos
SELECT
  p.title,
  p.property_type,
  COUNT(*)                       AS total_reservas,
  ROUND(SUM(f.total_amount), 2)  AS ingresos_totales
FROM gold.gold_fact_reservas f
JOIN gold.gold_dim_properties p
  ON f.property_id = p.property_key
GROUP BY p.title, p.property_type
ORDER BY ingresos_totales DESC
LIMIT 10;

## Conclusión Gold

La capa Gold quedó construida sobre Silver con el Star Schema completo:

- **Fact**: `gold.gold_fact_reservas` (granularidad: una fila por reserva)
- **Dims**: `gold_dim_users`, `gold_dim_properties`, `gold_dim_destinations`, `gold_dim_time`

El modelo permite responder preguntas analíticas sobre ingresos, ocupación,
desempeño de propiedades y comportamiento geográfico, y está listo para
ser conectado al SQL Warehouse de Databricks desde Power BI.